In [3]:
# %pip install langchain_community langchain neo4j langchain-huggingface ipywidgets einops pypdf tiktoken

In [1]:
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
# Initialize the HuggingFace embeddings model with specific parameters
embeddings_model = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1.5",
    model_kwargs={"device": "cuda", "trust_remote_code": True},
)

<All keys matched successfully>


In [3]:
# Define Neo4j database connection parameters
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "yourpassword"

# Initialize the Neo4jGraph object with the connection parameters
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

# Create a Neo4jVector object from the existing graph database
vecFromGraphDB = Neo4jVector.from_existing_graph(
    embedding=embeddings_model,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name="vecFromGraph",
    node_label="GovernmentDocument",
    text_node_properties=["summary"],
    embedding_node_property="embedding",
)

In [4]:
# Perform a similarity search on the vector database with a query
simResults = vecFromGraphDB.similarity_search_with_relevance_scores("What are FPI's")
print(simResults[0][0].metadata["name"])

A.P. (DIR Series) Circular No. 24


In [5]:
# Define a Cypher query to find nodes connected to a specific document
query = "MATCH (start {name: 'A.P. (DIR Series) Circular No. 26'})<-[r]-(connected) RETURN connected"
query_results = graph.query(query)
query_results[0]["connected"]["name"]

'A.P. (DIR Series) Circular No. 31'